# 02_pruning

Structured channel pruning for DeclipNet deployment optimization.

1. Load best checkpoint (`weighted_l1_dwt`), verify baseline test SI-SDR / PESQ using OLA+routed inference
2. Apply structured channel pruning to all Conv1d layers at sparsity levels [20%, 40%, 60%, 80%]
3. Evaluate SI-SDR / PESQ at each level, report in summary table
4. Optional: short post-prune fine-tune (~10-20 epochs) to recover quality
5. Save best pruned model (pruning made permanent) to `02_deployment/models/`

In [1]:
import sys
import json
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
import numpy as np
from pathlib import Path
from pesq import pesq as pesq_fn

sys.path.insert(0, "..")
sys.path.insert(0, "../01_dnn")

from config import *
from util import DeclipNet, DeclipDataset, si_sdr, make_loss_fn

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {DEVICE}")

# OLA inference constants
FC_BYPASS = 0.015
HOP = BS // 2
WINDOW = torch.hann_window(BS, device=DEVICE)

# Model hyperparams (match training)
H = 8
N_ATTN = 3
NUM_HEADS = 4
FFN_DIM = 256

MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)

Device: mps


In [2]:
# Load best checkpoint
model = DeclipNet(H=H, N=N_ATTN, num_heads=NUM_HEADS, ffn_dim=FFN_DIM).to(DEVICE)
state_dict = torch.load(FINAL_OUT / "weighted_l1_dwt" / "best_model.pt", weights_only=True)
state_dict = {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}
model.load_state_dict(state_dict)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"DeclipNet loaded — {n_params:,} parameters")

# Load test manifest
with open(TEST_OUT / "test_manifest.json") as f:
    test_manifest = json.load(f)
print(f"Test set: {len(test_manifest)} utterances")

DeclipNet loaded — 314,001 parameters
Test set: 100 utterances


In [3]:
from deploy_util import eval_ola_routed

baseline = eval_ola_routed(model, test_manifest, DEVICE)
print(f"\n--- Baseline (unpruned) ---")
print(f"SI-SDR: {baseline['mean_sdr']:.4f} dB  |  PESQ: {baseline['mean_pesq']:.4f}")

utt 000 | SDR: 17.52 -> 27.56 dB (delta: 10.04) | PESQ: 2.597 -> 3.633
utt 001 | SDR: 3.85 -> 12.75 dB (delta: 8.90) | PESQ: 1.199 -> 2.415
utt 002 | SDR: 10.45 -> 19.69 dB (delta: 9.25) | PESQ: 1.521 -> 2.969
utt 003 | SDR: 25.96 -> 35.97 dB (delta: 10.00) | PESQ: 3.969 -> 4.274
utt 004 | SDR: 27.07 -> 37.28 dB (delta: 10.21) | PESQ: 3.749 -> 4.178
utt 005 | SDR: 9.72 -> 23.83 dB (delta: 14.11) | PESQ: 2.258 -> 3.974
utt 006 | SDR: 25.82 -> 33.50 dB (delta: 7.68) | PESQ: 4.140 -> 4.404
utt 007 | SDR: 35.82 -> 44.25 dB (delta: 8.43) | PESQ: 4.282 -> 4.479
utt 008 | SDR: 2.36 -> 10.64 dB (delta: 8.29) | PESQ: 1.198 -> 2.403
utt 009 | SDR: 32.38 -> 43.18 dB (delta: 10.80) | PESQ: 4.272 -> 4.485
utt 010 | SDR: 21.63 -> 26.19 dB (delta: 4.56) | PESQ: 4.147 -> 4.354
utt 011 | SDR: 12.66 -> 18.41 dB (delta: 5.75) | PESQ: 3.448 -> 4.163
utt 012 | SDR: 7.63 -> 17.32 dB (delta: 9.69) | PESQ: 2.254 -> 3.673
utt 013 | SDR: 44.83 -> 44.83 dB (delta: 0.00) | PESQ: 4.627 -> 4.627
utt 014 | SDR: 12.2

## Structured Channel Pruning

Apply `ln_structured` (L2-norm) pruning to all `Conv1d` weight tensors at increasing sparsity levels.
At each level, evaluate the pruned model (without any fine-tuning) to measure quality degradation.

In [5]:
import copy

SPARSITY_LEVELS = [0.05, 0.1, 0.15, 0.2]

def apply_structured_pruning(model, sparsity):
    """Apply L2-norm structured (channel) pruning to all Conv1d layers."""
    for name, module in model.named_modules():
        if isinstance(module, nn.Conv1d):
            prune.ln_structured(module, name="weight", amount=sparsity, n=2, dim=0)

def count_nonzero_params(model):
    total = 0
    nonzero = 0
    for p in model.parameters():
        total += p.numel()
        nonzero += (p != 0).sum().item()
    return nonzero, total

results = []

for sparsity in SPARSITY_LEVELS:
    print(f"\n{'='*60}")
    print(f"Sparsity: {sparsity:.0%}")
    print(f"{'='*60}")

    pruned_model = copy.deepcopy(model)
    pruned_model.eval()
    apply_structured_pruning(pruned_model, sparsity)

    nonzero, total = count_nonzero_params(pruned_model)
    print(f"Parameters: {nonzero:,} / {total:,} nonzero ({100*nonzero/total:.1f}%)")

    metrics = eval_ola_routed(pruned_model, test_manifest, DEVICE, verbose=False)

    print(f"SI-SDR: {metrics['mean_sdr']:.4f} dB  |  PESQ: {metrics['mean_pesq']:.4f}")
    print(f"  vs baseline — SI-SDR delta: {metrics['mean_sdr'] - baseline['mean_sdr']:.4f} dB  |  "
          f"PESQ delta: {metrics['mean_pesq'] - baseline['mean_pesq']:.4f}")

    results.append({
        "sparsity": sparsity,
        "nonzero": nonzero,
        "total": total,
        "mean_sdr": metrics["mean_sdr"],
        "mean_pesq": metrics["mean_pesq"],
        "sdr_delta": metrics["mean_sdr"] - baseline["mean_sdr"],
        "pesq_delta": metrics["mean_pesq"] - baseline["mean_pesq"],
    })

    del pruned_model


Sparsity: 5%
Parameters: 314,001 / 314,001 nonzero (100.0%)
SI-SDR: 14.7974 dB  |  PESQ: 3.0402
  vs baseline — SI-SDR delta: -12.5249 dB  |  PESQ delta: -0.9056

Sparsity: 10%
Parameters: 314,001 / 314,001 nonzero (100.0%)
SI-SDR: -3.7591 dB  |  PESQ: 1.7452
  vs baseline — SI-SDR delta: -31.0814 dB  |  PESQ delta: -2.2006

Sparsity: 15%
Parameters: 314,001 / 314,001 nonzero (100.0%)
SI-SDR: -17.7277 dB  |  PESQ: 1.5768
  vs baseline — SI-SDR delta: -45.0500 dB  |  PESQ delta: -2.3689

Sparsity: 20%
Parameters: 314,001 / 314,001 nonzero (100.0%)
SI-SDR: -20.4300 dB  |  PESQ: 1.6162
  vs baseline — SI-SDR delta: -47.7523 dB  |  PESQ delta: -2.3295


## Findings

Structured channel pruning (`ln_structured`, L2-norm, `dim=0`) is **not viable** for this model:

| Sparsity | SI-SDR (dB) | PESQ | SI-SDR delta |
|----------|-------------|------|--------------|
| 0% (baseline) | 27.32 | 3.95 | — |
| 5% | 14.80 | 3.04 | −12.52 |
| 10% | −3.76 | 1.75 | −31.08 |
| 15% | −17.73 | 1.58 | −45.05 |
| 20% | −20.43 | 1.62 | −47.75 |

- U-Net skip connections couple encoder/decoder channel dims — pruning one side without the other breaks the signal path
- `ln_structured` treats each layer independently, doesn't account for this coupling
- H=8 base channels means even one pruned channel is a 12.5% cut at the narrowest point — no headroom
- Model is already 314k params; more advanced pruning not worth it here
- Better to focus on quantization and efficient export (ONNX, CoreML) for deployment gains